# `UCN.download()`: choosing data, what you get back, and how it performs

The Unified Ceilometer Network (UCN, Hampton University) publishes **one file per ceilometer per UTC day**. atmoz reads
**L2** (the default: 10-minute backscatter profiles and boundary-layer / cloud retrievals, ~1.4 MB a day) and **L3** (hourly statistics).

This notebook shows:

1. **How to select the data you want** (sections 1 and 3)
2. **What `download()` returns**: DataFrames with one row per time step (section 2)
3. **Plots** (section 4)
4. **Speed**: `pull -> download -> parse -> transform -> return -> slice` (section 5)
5. **Large pulls**: how atmoz protects the UCN server and your RAM (section 6)

> **`LIVE`** (next cell). `True` contacts the UCN server; `False` never does and uses only what is already in the atmoz cache.
> Cells that need the network are marked **[network]**. Files are ~1.4 MB each and are downloaded **once**; re-runs come from the cache.

Install: `pip install -e ".[lidar]" jupyter` from the repo root.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

import atmoz
from atmoz.lidar import UCN
from atmoz.exceptions import RequestTooLargeError

LIVE = True                 # False = never touch the network
CEILO = 9                   # McMillan Reservoir, DC (chosen by name in section 1)
DAY = "2026-11-01"          # a UTC day at least a few days old, so L2 exists and is final
MULTI_DAYS = 3              # days stacked in sections 4 and 5e
SCALING_DAYS = [1, 3]       # scaling test in section 5c: re-downloads sum(days) x 2 files (~1.4 MB each)

def timeit(fn, n=1):
    """Run fn() n times; return (last result, best time in seconds)."""
    best, out = float("inf"), None
    for _ in range(n):
        t = time.perf_counter(); out = fn(); best = min(best, time.perf_counter() - t)
    return out, best

print("atmoz", atmoz.__version__, "| cache:", atmoz.cache_dir("ucn"))

## 1. How users select the data they want

A request is **four choices**:

| Choice | Argument | Example |
|---|---|---|
| Which ceilometer(s) | `ceilometer` (number or list) | `9`, `[9, 5]` |
| Which UTC days | `start_date`, `end_date` (inclusive; `end_date` defaults to `start_date`) | `"2026-11-01"`, `"2026-11-07"` |
| Which product | `level` | `"L2"` (default), `"L3"`, or both |
| What to do about gaps | `skip_missing` | `True` leaves out days with no file |

Everything else (time of day, height range, variables) is chosen **after** the download by slicing the DataFrame (section 3): a day is only ~1.2 MB in memory.

### 1a. Find the ceilometer number: `UCN.sites()`

In [ ]:
if LIVE:                    # [network] one small page
    sites = UCN.sites()
else:                       # offline sample (a few real rows)
    sites = pd.DataFrame({"ceilometer": [1, 2, 4, 5, 9], "site": ["UMBC", "Unknown Site", "CCNY", "Edgewood, MD", "McMillan Reservoir, DC"]})
print(len(sites), "ceilometers")
sites.head(12)

In [ ]:
hits = sites[sites["site"].str.contains("McMillan|Edgewood", case=False)]     # search by name, case-insensitive
display(hits)
CEILO = int(sites.loc[sites["site"].str.contains("McMillan", case=False), "ceilometer"].iloc[0])
print("chosen ceilometer:", CEILO)

> **Not available yet:** selecting by **state, region or bounding box**. `sites()` returns only the number and the name; coordinates only appear
> inside each file (`df.attrs["latitude"]`, `["longitude"]`) after it has been downloaded.

### 1b. Preview a request before downloading: `UCN.available()`
It moves **no data**. It lists one row per (ceilometer, day, level) with an estimated size, whether the UCN viewer lists that day, and whether it is already cached.

In [ ]:
if LIVE:                    # [network] one 40 KB page per ceilometer (cached 6 h)
    plan = UCN.available([CEILO, 5], "2026-10-30", "2026-11-01", level=["L2", "L3"])
else:
    plan = pd.DataFrame(UCN._validate([CEILO, 5], "2026-10-30", "2026-11-01", level=["L2", "L3"]))
display(plan)
if LIVE:
    print(f"would download about {plan['est_download_mb'].sum():.1f} MB ({int((~plan['cached']).sum())} files not cached yet)")

## 2. What `download()` returns

In [ ]:
ds, t = timeit(lambda: UCN.download(CEILO, DAY, silent=True))
print(type(ds).__name__, "| datatype:", ds.datatype, "| took", f"{t:.2f}s")
print("metadata:", ds.metadata)
name, df = next(iter(ds.data.items()))
print("data keys:", list(ds.data), "| DataFrame", df.shape)
df.iloc[:4, :6]

In [ ]:
# One row per 10 minutes; plain columns hold numbers, "list" columns hold one numpy array per row
kinds = pd.DataFrame({
    "kind": ["list of %d" % len(df[c].iloc[0]) if isinstance(df[c].iloc[0], np.ndarray) else "number" for c in df.columns],
    "dtype": [str(df[c].iloc[0].dtype) if isinstance(df[c].iloc[0], np.ndarray) else str(df[c].dtype) for c in df.columns],
    "units": [df.attrs["units"].get(c, "") for c in df.columns],
})
print(len(df.columns), "columns,", int((kinds["kind"] != "number").sum()), "of them list columns")
kinds

In [ ]:
# Heights, labels, units and the file's own attributes travel with the DataFrame in df.attrs
print("keys:", list(df.attrs))
print("ceilometer", df.attrs["ceilometer"], "| lat/lon", round(df.attrs["latitude"], 4), round(df.attrs["longitude"], 4))
print("heights:", len(df.attrs["altitude_m"]), "gates,", df.attrs["altitude_m"][0], "to", df.attrs["altitude_m"][-1], "m")
print("labels of the 2-D columns:", df.attrs["list_labels"])
pd.Series(df.attrs["attrs"])

## 3. Selecting inside the data (slicing)
Everything is in memory after `download()`, so slicing is instant. Times are **UTC**.

In [ ]:
t0, t1 = f"{DAY} 12:00", f"{DAY} 18:00"
print("whole day            :", df.shape)
print("time window (6 h)    :", df.loc[t0:t1].shape)
print("one variable         :", df["nsl_ml_height"].shape, "->", df["nsl_ml_height"].dropna().shape, "non-missing")
prof = df["attenuated_backscatter"].loc[pd.Timestamp(f"{DAY} 15:00")]            # one profile (a numpy array)
print("one profile          :", prof.shape, prof.dtype)
da = UCN.backscatter(df)                                                          # (time, altitude) xarray DataArray
print("as an array          :", dict(da.sizes))
print("height 0-3 km        :", dict(da.sel(altitude=slice(0, 3000)).sizes))
print("time + height        :", dict(da.sel(time=slice(t0, t1), altitude=slice(0, 3000)).sizes))
print("hourly mean of NSL/ML:", df["nsl_ml_height"].resample("1h").mean().shape)

In [ ]:
# Select by LOCAL time: convert to a UTC window first (DST handled by the time zone database)
def utc_window(day, start_local, end_local, tz="America/New_York"):
    a = pd.Timestamp(f"{day} {start_local}", tz=tz).tz_convert("UTC").tz_localize(None)
    b = pd.Timestamp(f"{day} {end_local}", tz=tz).tz_convert("UTC").tz_localize(None)
    return a, b

a, b = utc_window(DAY, "06:00", "18:00")
daytime = df.loc[a:b]
print(f"06:00-18:00 local = {a} -> {b} UTC: {len(daytime)} rows")

## 4. Plot it

In [ ]:
da = UCN.backscatter(df)
pos = da.values[da.values > 0]
lo, hi = float(np.percentile(pos, 5)), float(np.percentile(da.sel(altitude=slice(0, 3000)).values, 99.5))
print("backscatter units:", da.attrs["units"], "| colour range:", f"{lo:.3g} .. {hi:.3g}", "| share <= 0:", f"{float((da.values <= 0).mean()):.1%}")

def curtain(ax, da, vmin, vmax, zmax_km=None):
    da = da.transpose("altitude", "time")
    m = ax.pcolormesh(da["time"].values, da["altitude"].values / 1000, da.clip(min=vmin).values,
                      norm=LogNorm(vmin, vmax), cmap="turbo", shading="auto")
    ax.set_ylabel("height (km)")
    if zmax_km: ax.set_ylim(0, zmax_km)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    return m

fig, ax = plt.subplots(figsize=(11, 4.5))
m = curtain(ax, da, lo, hi, zmax_km=4)
fig.colorbar(m, ax=ax, label=f"attenuated backscatter ({da.attrs['units']})")
ax.plot(df.index, df["nsl_ml_height"] / 1000, "o", ms=4, mfc="yellow", mec="k", mew=0.4, label="NSL / ML height")
ax.plot(df.index, df["rl_height"] / 1000, "s", ms=4, mfc="magenta", mec="k", mew=0.4, label="residual layer")
ax.plot(df.index, df["cloud_base_proxy"] / 1000, "^", ms=4, mfc="white", mec="k", mew=0.4, label="cloud-base proxy")
ax.legend(loc="upper right", fontsize=8); ax.set_xlabel("UTC")
ax.set_title(f"UCN L2, ceilometer {CEILO}, {DAY}: backscatter and boundary-layer heights")
fig.tight_layout()

In [ ]:
# One profile and the day's mean profile
fig, ax = plt.subplots(figsize=(4.5, 4.5))
z = np.asarray(df.attrs["altitude_m"]) / 1000
ax.plot(da.sel(time=f"{DAY} 15:00", method="nearest"), z, label="15:00 UTC")
ax.plot(da.mean("time"), z, label="day mean")
ax.set_xscale("symlog", linthresh=100); ax.set_ylim(0, 4); ax.set_xlabel(da.attrs["units"]); ax.set_ylabel("height (km)"); ax.legend()
fig.tight_layout()

In [ ]:
# L3: UCN's own hourly statistics (a small table; its backscatter array duplicates L2's and is not stored)
l3 = UCN.download(CEILO, DAY, level="L3", silent=True).data[f"CEILO-{CEILO}_{DAY.replace('-', '')}_L3"]
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.errorbar(l3.index, l3["nsl_ml_hourly_mean"] / 1000, yerr=l3["nsl_ml_hourly_std"] / 1000, fmt="o-", color="tab:orange", capsize=2, label="NSL / ML")
ax.errorbar(l3.index, l3["rl_hourly_mean"] / 1000, yerr=l3["rl_hourly_std"] / 1000, fmt="s-", color="tab:purple", capsize=2, label="residual layer")
ax.set_ylabel("height (km)"); ax.legend(fontsize=8); ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M")); ax.set_title("L3 hourly means")
fig.tight_layout()
print(l3.shape, "| cached L3 file:", round((atmoz.cache_dir("ucn") / f"CEILO-{CEILO}_{DAY.replace('-', '')}_L3.parquet").stat().st_size / 1e3, 1), "KB")

## 5. Speed tests: where does the time go?

`download()` is five private steps. We time each on **one L2 file**:

| # | Step | What it does |
|---|------|--------------|
| 1 | `_validate` | request -> jobs |
| 2 | `_fetch_one` | make sure the compact parquet is in the cache (a cache hit skips the network) |
| 3 | `_parse_one` | read the parquet into a DataFrame |
| 4 | `_assemble` | wrap in an `atmoz_dataset` and return it |

The first time a day is fetched there is extra work: **network** (the NetCDF comes down once), then a one-off **convert** to parquet. Section 5b times that.

### 5a. Stage breakdown (file already cached, so no network)

In [ ]:
job = UCN._validate(CEILO, DAY)[0]
cache = atmoz.cache_dir("ucn")
stages = {}
_, stages["1 validate"]                    = timeit(lambda: UCN._validate(CEILO, DAY), n=20)
path, stages["2 fetch (cache hit)"]        = timeit(lambda: UCN._fetch_one(job, dest_dir=cache, pause=0), n=5)
df1, stages["3 parse: parquet -> DataFrame"] = timeit(lambda: UCN._parse_one(job, path), n=5)
_, stages["4 assemble + return"]           = timeit(lambda: UCN._assemble({UCN._name(job): df1}), n=20)
_, total                                   = timeit(lambda: UCN.download(CEILO, DAY, silent=True, pause=0), n=5)

tbl = pd.DataFrame({"ms": {k: v * 1000 for k, v in stages.items()}}).round(2)
tbl.loc["sum of steps"] = tbl["ms"].sum().round(2)
tbl.loc["download() end to end"] = round(total * 1000, 2)
mem = df1.memory_usage(deep=True).sum() + sum(a.nbytes for a in df1["attenuated_backscatter"])
print(f"cached file: {path.stat().st_size/1e6:.2f} MB on disk | DataFrame: about {mem/1e6:.2f} MB in memory | UCN sends ~1.4 MB (NetCDF)")
tbl

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.barh(list(stages)[::-1], [v * 1000 for v in stages.values()][::-1], color="tab:blue")
ax.set_xlabel("milliseconds (best of n, warm cache)"); ax.set_title("download() stage breakdown"); fig.tight_layout()

### 5b. The first time: network vs convert **[network]**
`refresh=True` forces one fresh download of this ~1.4 MB file. We time the one-off convert-to-parquet step separately.

In [ ]:
if LIVE:
    spent, orig = {}, UCN.__dict__["_convert"].__func__
    def timed(cls, job, nc_path, dest):
        t = time.perf_counter(); orig(cls, job, nc_path, dest); spent["convert"] = time.perf_counter() - t
    UCN._convert = classmethod(timed)
    try:
        _, t_first = timeit(lambda: UCN._fetch_one(job, dest_dir=cache, refresh=True, pause=0))
    finally:
        UCN._convert = classmethod(orig)
    print(f"first fetch of a day : {t_first:6.2f} s")
    print(f"  network + write    : {t_first - spent['convert']:6.2f} s")
    print(f"  convert to parquet : {spent['convert']:6.3f} s   (one-off, done once per day)")
    _, t_warm = timeit(lambda: UCN.download(CEILO, DAY, silent=True, pause=0), n=5)
    print(f"download() warm      : {t_warm*1000:6.1f} ms  -> {t_first / t_warm:.0f}x faster than the first time")
else:
    print("LIVE = False: skipped")

### 5c. Scaling: more days, more workers **[network]**
Defaults are deliberately gentle (2 workers, 0.5 s between request starts). This **re-downloads** the files, so `SCALING_DAYS` is small.

In [ ]:
if LIVE:
    rows = []
    for nd in SCALING_DAYS:
        start = (pd.Timestamp(DAY) - pd.Timedelta(days=nd - 1)).date()
        for w in (1, 2):
            _, t = timeit(lambda: UCN.download(CEILO, start, DAY, refresh=True, skip_missing=True, max_workers=w, silent=True))
            rows.append({"days": nd, "workers": w, "seconds": round(t, 2), "s_per_day": round(t / nd, 2)})
    display(pd.DataFrame(rows).pivot(index="days", columns="workers", values="seconds").rename_axis(columns="workers -> seconds"))
else:
    print("LIVE = False: skipped")

### 5d. Slicing: in memory vs reading the cached file again

In [ ]:
import pyarrow.parquet as pq
t0, t1 = f"{DAY} 12:00", f"{DAY} 18:00"
ops_mem = {
    "pick a column":            lambda: df1["nsl_ml_height"],
    "time window (6 h)":        lambda: df1.loc[t0:t1],
    "one profile":              lambda: df1["attenuated_backscatter"].iloc[90],
    "backscatter as an array":  lambda: UCN.backscatter(df1),
    "array, 0-3 km, 6 h":       lambda: UCN.backscatter(df1).sel(time=slice(t0, t1), altitude=slice(0, 3000)),
    "hourly mean of a column":  lambda: df1["nsl_ml_height"].resample("1h").mean(),
}
ops_disk = {
    "read one small column":    lambda: pq.read_table(path, columns=["nsl_ml_height"]),
    "read the whole file":      lambda: UCN._parse_one(job, path),
}
rows = [{"operation": k, "where": "in memory", "ms": round(timeit(f, n=5)[1] * 1000, 3)} for k, f in ops_mem.items()]
rows += [{"operation": k, "where": "from the cache", "ms": round(timeit(f, n=5)[1] * 1000, 2)} for k, f in ops_disk.items()]
pd.DataFrame(rows).set_index("operation")

### 5e. Several days: `UCN.stack()`
`download()` returns one DataFrame per day. `stack()` joins them into one continuous, time-sorted DataFrame.

In [ ]:
start = (pd.Timestamp(DAY) - pd.Timedelta(days=MULTI_DAYS - 1)).date()
multi, t = timeit(lambda: UCN.download(CEILO, start, DAY, skip_missing=True, silent=True))
print(f"{len(multi.data)} files in {t:.2f}s | missing: {multi.metadata['missing']}")
long = UCN.stack(multi, CEILO)
print("stacked:", long.shape, "| days:", long.attrs["dates"])
da_long = UCN.backscatter(long)
fig, ax = plt.subplots(figsize=(11, 3.5))
m = curtain(ax, da_long, lo, hi, zmax_km=4)
fig.colorbar(m, ax=ax, label="backscatter"); ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %Hh"))
ax.set_title(f"Ceilometer {CEILO}, {start} to {DAY}"); fig.tight_layout()

## 6. Large pulls: protecting the server and your RAM

What things cost (measured on the file above, extended arithmetically):

In [ ]:
disk_mb, mem_mb = path.stat().st_size / 1e6, mem / 1e6
dl_mb = 1.4                                            # what UCN sends per L2 day
scen = [("1 ceilometer, 1 week", 1, 7), ("1 ceilometer, 1 year", 1, 365), ("1 ceilometer, full history (2,500 days)", 1, 2500),
        ("all 67 ceilometers, 1 month", 67, 30), ("all 67 ceilometers, 1 year", 67, 365)]
pd.DataFrame([{
    "pull": label, "files": c * d, "download GB": round(c * d * dl_mb / 1000, 2),
    "cache GB": round(c * d * disk_mb / 1000, 2), "RAM GB (all in memory)": round(c * d * mem_mb / 1000, 2),
    "time at 1 file / 1.2 s (default pacing), min": round(c * d * 1.2 / 60, 1),
} for label, c, d in scen]).set_index("pull")

In [ ]:
# A request that is too big stops BEFORE any request is made (this cell never touches the network):
try:
    UCN.download(list(range(1, 68)), "2025-01-01", "2025-12-31")      # every ceilometer, a whole year
except RequestTooLargeError as e:
    print(type(e).__name__)
    print(e)

The protections, in one place:

| Protection | What it does |
|---|---|
| **Download once** | the cache keeps one parquet file per day (about half the NetCDF size); days older than 2 are final and never re-downloaded |
| **Gentle pacing** | 2 workers, `pause=0.5` s between request starts, retries with backoff, `atmoz/<version>` User-Agent |
| **Skip what is not there** | `skip_missing=True` uses the viewer's list of available days, so unlisted days are never requested |
| **Size guard** | more than ~1 GB to download (~700 days) or ~2 GB in memory raises `RequestTooLargeError` first; `allow_large=True` overrides |
| **Preview** | `UCN.available(...)` shows files, size and what is cached before anything moves |
| **Disk only** | `UCN.fetch(...)` fills the cache and returns parquet paths without holding anything in memory |

## 7. Cache and cleanup

In [ ]:
print(atmoz.cache_info())          # {"ucn": {"files": N, "bytes": B}}
# atmoz.clear_cache("ucn")         # delete only the UCN files

## Summary: how a user selects data

| I want | Do this |
|--------|---------|
| a ceilometer by name | `UCN.sites()`, filter the `site` text, take `ceilometer` |
| to see what a request costs first | `UCN.available(9, "2026-10-01", "2026-10-31")` |
| one day | `UCN.download(9, "2026-11-01")` |
| a date range, several ceilometers or levels | `UCN.download([9, 5], "2026-10-30", "2026-11-01", level=["L2", "L3"], skip_missing=True)` |
| only the files, no parsing | `UCN.fetch(...)` returns `{name: parquet path}` |
| a time of day, a variable | slice the DataFrame: `df.loc["2026-11-01 12:00":"2026-11-01 18:00", "nsl_ml_height"]` |
| a height range | `UCN.backscatter(df).sel(altitude=slice(0, 3000))` |
| one continuous multi-day table | `UCN.stack(ds, 9)` |
| a state, region or bounding box | **not available yet**: `sites()` has no coordinates |